# KDE Estimaciones

Este notebook estima KDE sobre los valores OTU positivos. Primero compara `cv`, `scott` y `silverman` como opciones de bandwidth; despues usa la opcion elegida para generar las figuras KDE con los seis kernels.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "kernels").exists():
    KERNEL_TESTS_ROOT = cwd
elif (cwd / "Kernel_Tests" / "kernels").exists():
    KERNEL_TESTS_ROOT = cwd / "Kernel_Tests"
else:
    raise FileNotFoundError("No se pudo localizar la carpeta Kernel_Tests/kernels.")

PROJECT_ROOT = KERNEL_TESTS_ROOT.parent
FIGURES_DIR = KERNEL_TESTS_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(KERNEL_TESTS_ROOT) not in sys.path:
    sys.path.insert(0, str(KERNEL_TESTS_ROOT))

from kernels import COLOR_MAP, KERNELS
from kernels.bandwidth import cv_loglik, scott_h, silverman_h
from kernels.data import load_otu_positives
from kernels.kde import KDEEvaluator, gpu_available
from kernels.stats import mode_kde, normalize_conditional

print(f"Kernel_Tests: {KERNEL_TESTS_ROOT}")
print(f"Figuras: {FIGURES_DIR}")

In [ ]:
def suggest_grid_size(n_values):
    if n_values <= 250_000:
        return 1000
    if n_values <= 1_000_000:
        return 1500
    return 2000


def make_log_grid(values, grid_size, bandwidth):
    values = np.asarray(values, dtype=float)
    positive_min = float(np.min(values[values > 0]))
    lower = max(positive_min * 1e-3, 1e-12)
    upper = float(np.max(values) + 8.0 * bandwidth)
    return np.logspace(np.log10(lower), np.log10(upper), int(grid_size))


def prompt_text(prompt, default, env_var=None):
    env_value = os.environ.get(env_var) if env_var else None
    if env_value is not None and str(env_value).strip() != "":
        return str(env_value).strip()
    try:
        raw = input(prompt)
    except Exception as exc:
        print(f"Entrada no disponible ({type(exc).__name__}); usando default={default!r}.")
        return default
    raw = raw.strip()
    return raw if raw else default


def choose_bandwidth_method(valid_methods, default="cv"):
    prompt = f"Elige bandwidth {tuple(valid_methods)} [default={default}]: "
    selected = prompt_text(prompt, default, env_var="KDE_BANDWIDTH_METHOD").lower()
    if selected not in valid_methods:
        print(f"Opcion invalida {selected!r}; usando {default!r}.")
        selected = default
    return selected


def choose_grid_size(default_grid):
    raw = prompt_text(f"Grid size [default={default_grid}]: ", str(default_grid), env_var="KDE_GRID_SIZE")
    try:
        grid = int(raw)
    except ValueError:
        print(f"Grid invalido {raw!r}; usando {default_grid}.")
        grid = int(default_grid)
    if grid < 100:
        print("Grid menor a 100; usando 100.")
        grid = 100
    return grid

In [ ]:
values = load_otu_positives(verbose=True)
requested_backend = os.environ.get("KDE_BACKEND", "auto").lower()
if requested_backend == "auto":
    backend = "gpu" if gpu_available() else "cpu"
elif requested_backend == "gpu" and gpu_available():
    backend = "gpu"
elif requested_backend == "gpu":
    print("KDE_BACKEND='gpu' solicitado, pero CUDA/CuPy no esta disponible; usando CPU.")
    backend = "cpu"
elif requested_backend == "cpu":
    backend = "cpu"
else:
    raise ValueError("KDE_BACKEND debe ser 'auto', 'gpu' o 'cpu'.")
grid_suggestion = suggest_grid_size(len(values))

print(f"Backend KDE: {backend}")
print(f"Grid sugerido por volumen de datos: {grid_suggestion}")
print(f"Kernels: {KERNELS}")

In [ ]:
CV_REFERENCE_KERNEL = os.environ.get("KDE_CV_KERNEL", "gaussian").lower()
CV_SUBSAMPLE = int(os.environ.get("KDE_CV_SUBSAMPLE", "5000"))
CV_FOLDS = int(os.environ.get("KDE_CV_FOLDS", "5"))
CV_BW_GRID = int(os.environ.get("KDE_CV_BW_GRID", "25"))

if CV_REFERENCE_KERNEL not in KERNELS:
    raise ValueError(f"KDE_CV_KERNEL debe estar en {KERNELS}.")

print("Estimando bandwidths...")
h_scott = scott_h(values)
h_silverman = silverman_h(values)
h_cv, cv_grid, cv_scores = cv_loglik(
    values,
    kernel=CV_REFERENCE_KERNEL,
    cv_folds=CV_FOLDS,
    n_subsample=CV_SUBSAMPLE,
    n_bw=CV_BW_GRID,
)

bandwidths = {
    "cv": h_cv,
    "scott": h_scott,
    "silverman": h_silverman,
}

bandwidth_df = pd.DataFrame(
    [{"method": method, "bandwidth": value} for method, value in bandwidths.items()]
)
print(f"CV kernel de referencia: {CV_REFERENCE_KERNEL}")
display(bandwidth_df)

In [ ]:
comparison_grid = make_log_grid(values, grid_suggestion, max(bandwidths.values()))

fig, ax = plt.subplots(figsize=(10, 5))
for method, bandwidth in bandwidths.items():
    pdf = KDEEvaluator(
        values,
        kernel=CV_REFERENCE_KERNEL,
        bandwidth=bandwidth,
        backend=backend,
    ).evaluate(comparison_grid)
    density, mass = normalize_conditional(pdf, comparison_grid)
    ax.plot(
        comparison_grid,
        density,
        linewidth=1.8,
        label=f"{method} (h={bandwidth:.3g}, masa={mass:.3f})",
    )

ax.set_xscale("log")
ax.set_xlabel("Valor OTU positivo (escala log)")
ax.set_ylabel("Densidad condicional")
ax.set_title(f"Comparacion de bandwidths KDE ({CV_REFERENCE_KERNEL})")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
fig.tight_layout()

bandwidth_fig = FIGURES_DIR / "bandwidth_comparison.png"
fig.savefig(bandwidth_fig, dpi=160, bbox_inches="tight")
print(f"Figura guardada: {bandwidth_fig}")
plt.show()

In [ ]:
selected_method = choose_bandwidth_method(bandwidths.keys(), default="cv")
selected_bandwidth = bandwidths[selected_method]
grid_size = choose_grid_size(grid_suggestion)
x_grid = make_log_grid(values, grid_size, selected_bandwidth)

print(f"Bandwidth elegido: {selected_method} -> h={selected_bandwidth:.6g}")
print(f"Grid final: {grid_size}")

In [ ]:
densities = {}
summary_rows = []

for kernel in KERNELS:
    pdf = KDEEvaluator(
        values,
        kernel=kernel,
        bandwidth=selected_bandwidth,
        backend=backend,
    ).evaluate(x_grid)
    density, mass = normalize_conditional(pdf, x_grid)
    densities[kernel] = density
    summary_rows.append({
        "kernel": kernel,
        "bandwidth_method": selected_method,
        "bandwidth_usado": selected_bandwidth,
        "masa": mass,
        "moda_kde": mode_kde(density, x_grid),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for kernel in KERNELS:
    ax.plot(
        x_grid,
        densities[kernel],
        color=COLOR_MAP[kernel],
        linewidth=1.8,
        label=kernel,
    )

ax.set_xscale("log")
ax.set_xlabel("Valor OTU positivo (escala log)")
ax.set_ylabel("Densidad condicional")
ax.set_title(f"KDE de los seis kernels (bandwidth={selected_method}, h={selected_bandwidth:.3g})")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()

all_kernels_fig = FIGURES_DIR / "kde_all_kernels.png"
fig.savefig(all_kernels_fig, dpi=160, bbox_inches="tight")
print(f"Figura guardada: {all_kernels_fig}")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for ax, kernel in zip(axes.ravel(), KERNELS):
    ax.plot(x_grid, densities[kernel], color=COLOR_MAP[kernel], linewidth=1.8)
    ax.set_xscale("log")
    ax.set_title(kernel)
    ax.grid(True, alpha=0.25)

for ax in axes[-1, :]:
    ax.set_xlabel("Valor OTU positivo")
for ax in axes[:, 0]:
    ax.set_ylabel("Densidad condicional")

fig.suptitle(f"KDE por kernel (bandwidth={selected_method}, grid={grid_size})", fontsize=13)
fig.tight_layout()

grid_fig = FIGURES_DIR / "kde_by_kernel_grid.png"
fig.savefig(grid_fig, dpi=160, bbox_inches="tight")
print(f"Figura guardada: {grid_fig}")
plt.show()